# Projeto AutoValor — Sistema de Previsão de Preço de Veículos## Análise Exploratória e Treinamento de Modelos**Disciplina:** Especialização em Data Science — UNICAMP  **Dataset:** Tabela FIPE — Histórico de Preços (Kaggle)  **Objetivo:** Desenvolver um pipeline de Machine Learning para prever o preço de referência FIPE de veículos a partir de suas características técnicas e comerciais.---

## Etapa 1 — Análise Exploratória de Dados (EDA)### 1.1 Carregamento e Inspeção InicialPrimeiro passo: carregar o dataset e inspecionar sua estrutura — tipos de dados, dimensões, valores nulos e estatísticas descritivas. É o equivalente a um "full scan" antes de montar o método analítico.

In [ ]:
import pandas as pd

# Carregamento do dataset
df = pd.read_csv(r"C:\Users\leogr\Downloads\archive (4)\tabela-fipe-historico-precos.csv")

# Inspeção inicial
print("Shape:", df.shape)
print("\n--- Colunas e tipos ---")
print(df.dtypes)
print("\n--- Primeiras 5 linhas ---")
print(df.head())
print("\n--- Estatísticas descritivas ---")
print(df.describe())
print("\n--- Valores nulos por coluna ---")
print(df.isnull().sum())

### 1.2 Análise Aprofundada das VariáveisInvestigação da cardinalidade das variáveis categóricas (marcas, modelos), distribuição das variáveis numéricas e faixas temporais do dataset.

In [ ]:
# Remover a coluna "Unnamed: 0" (índice antigo sem informação útil)
df = df.drop(columns=["Unnamed: 0"])

# Cardinalidade das variáveis categóricas
print("Marcas únicas:", df["marca"].nunique())
print("Modelos únicos:", df["modelo"].nunique())
print("Códigos FIPE únicos:", df["codigoFipe"].nunique())

# Top 10 marcas com mais registros
print("\n--- Top 10 marcas (por frequência) ---")
print(df["marca"].value_counts().head(10))

# Faixa temporal
print("\n--- Faixa de anoModelo ---")
print(f"De {df['anoModelo'].min()} a {df['anoModelo'].max()}")
print("\n--- Período de referência ---")
print(f"De {df['mesReferencia'].min()}/{df['anoReferencia'].min()} a {df['mesReferencia'].max()}/{df['anoReferencia'].max()}")

# Distribuição do valor — percentis para entender outliers
print("\n--- Percentis do valor (R$) ---")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p}: R$ {df['valor'].quantile(p/100):,.0f}")

### 1.3 Visualização das DistribuiçõesQuatro gráficos exploratórios: distribuição do preço bruto, distribuição em escala logarítmica (para avaliar normalidade), evolução do preço mediano por ano do modelo e por ano de referência FIPE.**Observação-chave:** A transformação logarítmica normaliza a distribuição altamente assimétrica do preço, indicando que predizer o log(preço) é mais adequado para os modelos de regressão.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histograma do preço
axes[0,0].hist(df["valor"], bins=100, color="steelblue", edgecolor="none")
axes[0,0].set_title("Distribuição de Preço (todos)")
axes[0,0].set_xlabel("Valor (R$)")
axes[0,0].axvline(df["valor"].median(), color="red", linestyle="--", label=f'Mediana: R${df["valor"].median():,.0f}')
axes[0,0].legend()

# 2. Histograma do LOG do preço
axes[0,1].hist(np.log1p(df["valor"]), bins=100, color="coral", edgecolor="none")
axes[0,1].set_title("Distribuição de log(Preço)")
axes[0,1].set_xlabel("log(Valor)")
axes[0,1].axvline(np.log1p(df["valor"].median()), color="red", linestyle="--", label="Mediana")
axes[0,1].legend()

# 3. Preço mediano por ano do modelo
preco_por_ano = df.groupby("anoModelo")["valor"].median()
axes[1,0].plot(preco_por_ano.index, preco_por_ano.values, marker="o", markersize=3, color="steelblue")
axes[1,0].set_title("Preço mediano por Ano do Modelo")
axes[1,0].set_xlabel("Ano do Modelo")
axes[1,0].set_ylabel("Valor mediano (R$)")

# 4. Preço mediano por ano de referência
preco_por_ref = df.groupby("anoReferencia")["valor"].median()
axes[1,1].plot(preco_por_ref.index, preco_por_ref.values, marker="s", markersize=5, color="coral")
axes[1,1].set_title("Preço mediano por Ano de Referência")
axes[1,1].set_xlabel("Ano de Referência")
axes[1,1].set_ylabel("Valor mediano (R$)")

plt.tight_layout()
plt.savefig("01_exploracao_precos.png", dpi=150)
plt.show()

### 1.4 Feature EngineeringCriação de três variáveis derivadas que capturam informações não explícitas no dataset original:1. **idade_veiculo**: diferença entre o ano de referência e o ano do modelo — captura a depreciação temporal2. **indice_marca**: preço mediano da marca dividido pelo preço mediano geral — indica posicionamento premium vs. popular (análogo a um "fator de enriquecimento")3. **variacao_preco_pct**: variação percentual do preço do modelo entre o registro mais antigo e o mais recente — indica tendência de valorização/depreciação

In [ ]:
# === FEATURE 1: Idade do veículo ===
df["idade_veiculo"] = df["anoReferencia"] - df["anoModelo"]

print("--- Idade do veículo ---")
print(f"Mín: {df['idade_veiculo'].min()}, Máx: {df['idade_veiculo'].max()}")
print(f"Idades negativas: {(df['idade_veiculo'] < 0).sum()} registros")

# === FEATURE 2: Índice de posicionamento de preço da marca ===
mediana_geral = df["valor"].median()
mediana_marca = df.groupby("marca")["valor"].median()
df["indice_marca"] = df["marca"].map(mediana_marca) / mediana_geral

print("\n--- Índice de posicionamento (top 10 premium) ---")
ranking = (mediana_marca / mediana_geral).sort_values(ascending=False)
print(ranking.head(10).round(2))

print("\n--- Índice de posicionamento (top 10 populares) ---")
print(ranking.tail(10).round(2))

# === FEATURE 3: Variação percentual de preço por modelo ===
def calc_variacao(grupo):
    grupo_sorted = grupo.sort_values(["anoReferencia", "mesReferencia"])
    preco_inicial = grupo_sorted["valor"].iloc[0]
    preco_final = grupo_sorted["valor"].iloc[-1]
    if preco_inicial > 0:
        return ((preco_final - preco_inicial) / preco_inicial) * 100
    return 0

variacao_modelo = df.groupby("codigoFipe").apply(calc_variacao)
variacao_modelo.name = "variacao_preco_pct"
df = df.merge(variacao_modelo.reset_index(), on="codigoFipe", how="left")

print("\n--- Variação % de preço por modelo ---")
print(df["variacao_preco_pct"].describe().round(2))

# === FEATURE BÔNUS: log do valor (variável alvo transformada) ===
df["log_valor"] = np.log1p(df["valor"])

print("\n--- Resumo das novas features ---")
print(df[["idade_veiculo", "indice_marca", "variacao_preco_pct", "log_valor"]].describe().round(2))

### 1.5 Investigação de Dados InconsistentesForam encontradas 110.014 idades negativas. Antes de descartar, investigamos exemplos concretos para entender a causa raiz.**Conclusão:** Idades de -1 são legítimas (modelo do ano seguinte já listado na FIPE, ex: modelo 2022 na tabela de 2021). Idades de -2 a -21 são artefatos de dados incorretos — por exemplo, um Alfa Romeo 156 (produzido até ~2007) aparecendo com anoModelo=2022 na FIPE de 2001, o que é fisicamente impossível.

In [ ]:
# Investigar idades negativas extremas
print("--- Exemplos com idade = -21 ---")
print(df[df["idade_veiculo"] == -21][["codigoFipe","marca","modelo","anoModelo","anoReferencia","mesReferencia","valor"]].head(10).to_string())

print("\n--- Exemplos com idade = -1 ---")
print(df[df["idade_veiculo"] == -1][["codigoFipe","marca","modelo","anoModelo","anoReferencia","mesReferencia","valor"]].head(10).to_string())

# Distribuição das idades negativas
print("\n--- Distribuição das idades negativas ---")
negativos = df[df["idade_veiculo"] < 0]["idade_veiculo"].value_counts().sort_index()
print(negativos)

### 1.6 Limpeza de Dados e Tratamento de OutliersDecisões de limpeza documentadas:1. **Idades impossíveis (< -1):** Removidos 102.261 registros com idades de -2 a -21, identificados como artefatos de encoding incorreto do anoModelo.2. **Outliers de preço:** Removidos registros abaixo do percentil 1 (R$ 4.832) e acima do percentil 99 (R$ 992.452), que representam veículos atípicos que distorceriam o modelo.3. **Recálculo das features:** Após a limpeza, as features derivadas (indice_marca, variacao_preco_pct) foram recalculadas sobre os dados limpos.**Resultado:** Dataset reduzido de 466.020 para 356.483 registros (~23% de perda), com ganho significativo em qualidade.

In [ ]:
import numpy as np

print(f"Antes da limpeza: {len(df)} registros")

# 1. Remover idades impossíveis (< -1)
df = df[df["idade_veiculo"] >= -1]
print(f"Após remover idade < -1: {len(df)} registros")

# 2. Tratar outliers de preço (P1 a P99)
p01 = df["valor"].quantile(0.01)
p99 = df["valor"].quantile(0.99)
print(f"\nLimites de preço: R${p01:,.0f} a R${p99:,.0f}")
df = df[(df["valor"] >= p01) & (df["valor"] <= p99)]
print(f"Após remover outliers de preço: {len(df)} registros")

# 3. Recalcular features sobre dados limpos
mediana_geral = df["valor"].median()
mediana_marca = df.groupby("marca")["valor"].median()
df["indice_marca"] = df["marca"].map(mediana_marca) / mediana_geral

def calc_variacao(grupo):
    grupo_sorted = grupo.sort_values(["anoReferencia", "mesReferencia"])
    preco_inicial = grupo_sorted["valor"].iloc[0]
    preco_final = grupo_sorted["valor"].iloc[-1]
    if preco_inicial > 0:
        return ((preco_final - preco_inicial) / preco_inicial) * 100
    return 0

variacao_modelo = df.groupby("codigoFipe").apply(calc_variacao, include_groups=False)
variacao_modelo.name = "variacao_preco_pct"
df = df.drop(columns=["variacao_preco_pct"])
df = df.merge(variacao_modelo.reset_index(), on="codigoFipe", how="left")
df["log_valor"] = np.log1p(df["valor"])

# Resumo final
print(f"\n--- Dataset limpo: resumo ---")
print(f"Registros: {len(df)}")
print(f"Marcas: {df['marca'].nunique()}")
print(f"Modelos: {df['modelo'].nunique()}")
print(f"Faixa de preço: R${df['valor'].min():,.0f} a R${df['valor'].max():,.0f}")
print(f"Faixa de idade: {df['idade_veiculo'].min()} a {df['idade_veiculo'].max()}")
print(f"\n--- Estatísticas do dataset limpo ---")
print(df.describe().round(2))

---## Etapa 2 — Treinamento e Validação de Modelos### 2.1 Preprocessamento para Machine LearningPreparação dos dados para alimentar os algoritmos:- **Target Encoding** da variável `marca`: cada marca é substituída pela média do log(preço) dos veículos daquela marca — uma codificação numérica que preserva a relação ordinal com o alvo.- **Seleção de features:** 7 variáveis numéricas (3 originais + 3 derivadas + 1 encoding).- **Divisão treino/teste:** 80%/20% com `random_state=42` para reprodutibilidade.- **Normalização (StandardScaler):** aplicada no treino e replicada no teste (sem data leakage — o `fit` é feito exclusivamente no treino).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Target Encoding para 'marca'
media_log_marca = df.groupby("marca")["log_valor"].mean()
df["marca_encoded"] = df["marca"].map(media_log_marca)

print("--- Target Encoding da marca (exemplos) ---")
exemplos_marcas = ["Ferrari", "Toyota", "VW - VolksWagen", "Lada", "Rolls-Royce"]
for m in exemplos_marcas:
    if m in media_log_marca.index:
        print(f"  {m}: {media_log_marca[m]:.2f} (= média de log_preço da marca)")

# Selecionar features
features = [
    "anoModelo", "mesReferencia", "anoReferencia",
    "idade_veiculo", "indice_marca", "variacao_preco_pct", "marca_encoded",
]

X = df[features].copy()
y = df["log_valor"].copy()

print(f"\n--- Matriz de features: {X.shape} ---")
print(f"Algum NaN? Features: {X.isnull().any().any()}, Alvo: {y.isnull().any()}")

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTreino: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}%) | Teste: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}%)")

# Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit somente no treino
X_test_scaled = scaler.transform(X_test)         # transform no teste

print(f"\nApós normalização — Médias: {X_train_scaled.mean(axis=0).round(4)}")
print(f"Após normalização — Desvios: {X_train_scaled.std(axis=0).round(4)}")
print("\n✓ Dados prontos para treinar os modelos!")

### 2.2 Treinamento e Comparação de 5 Algoritmos de RegressãoCinco modelos são treinados e avaliados:| Algoritmo | Descrição ||-----------|-----------|| **Regressão Linear** | Modelo linear clássico — baseline || **Ridge** | Regressão linear com regularização L2 para evitar overfitting || **Random Forest** | Ensemble de 200 árvores de decisão independentes || **Gradient Boosting** | Ensemble sequencial de 200 árvores, cada uma corrigindo os erros da anterior || **SVR** | Support Vector Regression com kernel RBF (amostra de 20% por custo computacional) |**Validação:** Cross-validation 5-fold no conjunto de treino (exceto SVR).  **Métricas:** RMSE, R² e MAE calculados no conjunto de teste (dados nunca vistos pelo modelo).

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

modelos = {
    "Regressão Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
    "SVR": SVR(kernel="rbf", C=10, epsilon=0.1),
}

resultados = {}

for nome, modelo in modelos.items():
    print(f"\n{'='*50}")
    print(f"Treinando: {nome}")
    inicio = time.time()
    
    if nome == "SVR":
        print("  (usando amostra de 20% para viabilidade computacional)")
        np.random.seed(42)
        idx = np.random.choice(len(X_train_scaled), size=int(len(X_train_scaled)*0.2), replace=False)
        modelo.fit(X_train_scaled[idx], y_train.iloc[idx])
        y_pred = modelo.predict(X_test_scaled)
    else:
        if nome in ["Regressão Linear", "Ridge"]:
            scores = cross_val_score(modelo, X_train_scaled, y_train, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
        else:
            scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
        
        rmse_cv = np.sqrt(-scores)
        print(f"  RMSE Cross-Val (5 folds): {rmse_cv.mean():.4f} ± {rmse_cv.std():.4f}")
        
        if nome in ["Regressão Linear", "Ridge"]:
            modelo.fit(X_train_scaled, y_train)
            y_pred = modelo.predict(X_test_scaled)
        else:
            modelo.fit(X_train, y_train)
            y_pred = modelo.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    erro_pct_aprox = (np.exp(rmse) - 1) * 100
    tempo = time.time() - inicio
    
    resultados[nome] = {"RMSE": rmse, "MAE": mae, "R²": r2, "Erro% aprox": erro_pct_aprox, "Tempo(s)": tempo}
    
    print(f"  RMSE (teste): {rmse:.4f}")
    print(f"  MAE  (teste): {mae:.4f}")
    print(f"  R²   (teste): {r2:.4f}")
    print(f"  Erro aprox:   ~{erro_pct_aprox:.1f}%")
    print(f"  Tempo: {tempo:.1f}s")

print(f"\n{'='*60}")
print("RESUMO COMPARATIVO — RANKING POR R²")
print(f"{'='*60}")
ranking = sorted(resultados.items(), key=lambda x: x[1]["R²"], reverse=True)
for i, (nome, m) in enumerate(ranking, 1):
    print(f"  {i}. {nome:25s} | R²={m['R²']:.4f} | RMSE={m['RMSE']:.4f} | ~{m['Erro% aprox']:.1f}% erro")

### 2.3 Importância das Features e Serialização do ModeloO Random Forest (R²=0.978) é o melhor modelo. A análise de importância das features revela que:- **marca_encoded** (35.5%) é a variável mais influente — a marca é o principal determinante do preço- As **3 features derivadas** (idade_veiculo, indice_marca, variacao_preco_pct) somam ~43% da importância total, validando o Feature Engineering- **mesReferencia** e **anoReferencia** são praticamente irrelevantes, indicando que o preço depende das características do veículo, não do momento da consultaO melhor modelo é serializado com `joblib` para uso na API REST (Etapa 3).

In [ ]:
import joblib

# Importância das features
importancias = modelos["Random Forest"].feature_importances_
feat_imp = sorted(zip(features, importancias), key=lambda x: x[1], reverse=True)

print("=== IMPORTÂNCIA DAS FEATURES (Random Forest) ===\n")
for feat, imp in feat_imp:
    barra = "█" * int(imp * 100)
    print(f"  {feat:25s} {imp:.4f}  {barra}")

# Gráfico de importância
fig, ax = plt.subplots(figsize=(10, 5))
nomes = [f[0] for f in feat_imp]
valores = [f[1] for f in feat_imp]
bars = ax.barh(nomes[::-1], valores[::-1], color="steelblue")
ax.set_xlabel("Importância relativa")
ax.set_title("Importância das Features — Random Forest (R²=0.978)")
for bar, val in zip(bars, valores[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("02_importancia_features.png", dpi=150)
plt.show()

# Salvar modelo e artefatos
artefatos = {
    "modelo": modelos["Random Forest"],
    "scaler": scaler,
    "features": features,
    "mediana_geral": mediana_geral,
    "media_log_marca": media_log_marca.to_dict(),
    "mediana_marca": mediana_marca.to_dict(),
}
joblib.dump(artefatos, "modelo_fipe.joblib")

import os
tamanho_mb = os.path.getsize("modelo_fipe.joblib") / (1024*1024)
print(f"\n✓ Modelo salvo em 'modelo_fipe.joblib' ({tamanho_mb:.1f} MB)")

# Teste de sanidade
print("\n=== TESTE: Predição de um Toyota modelo 2015 ===")
exemplo = {
    "anoModelo": 2015, "mesReferencia": 6, "anoReferencia": 2022,
    "idade_veiculo": 7,
    "indice_marca": mediana_marca.get("Toyota", mediana_geral) / mediana_geral,
    "variacao_preco_pct": -20.0,
    "marca_encoded": media_log_marca.get("Toyota", 10.0),
}
exemplo_df = pd.DataFrame([exemplo])[features]
log_pred = modelos["Random Forest"].predict(exemplo_df)[0]
preco_pred = np.expm1(log_pred)
print(f"  Preço estimado: R$ {preco_pred:,.0f}")